# Etapa 2 — Extracción, limpieza y transformación de datos
### Reto Global Classroom — Analítica de Airbnb | Ciudad: Austin, TX
**Rol:** Ingeniería de datos
**Fuente de datos:** Inside Airbnb — `listings.csv` (Austin, TX)

Este cuaderno documenta el análisis exploratorio inicial y el proceso de limpieza, transformación y preparación del conjunto de datos de Airbnb para Austin, de cara a las etapas posteriores del proyecto (análisis exploratorio, modelos de regresión/clasificación y tablero).

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 50)

RAW_PATH = "listings.csv"
df = pd.read_csv(RAW_PATH, low_memory=False)
df.shape

(11111, 90)

## 1. Análisis exploratorio de los datos iniciales
Número de variables, tipos de dato y proporción de nulos por columna.

In [2]:
print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])
print()
print("Tipos de dato:")
print(df.dtypes.value_counts())

Número de filas: 11111
Número de columnas: 90

Tipos de dato:
float64    37
str        28
int64      25
Name: count, dtype: int64


In [3]:
null_pct = (df.isnull().mean() * 100).round(1).sort_values(ascending=False)
null_pct[null_pct > 0].to_frame("% nulos")

,% nulos
neighborhood_overview,100.0
host_since,100.0
host_response_time,100.0
host_thumbnail_url,100.0
host_acceptance_rate,100.0
host_response_rate,100.0
host_verifications,100.0
neighbourhood,100.0
host_total_listings_count,100.0
host_neighbourhood,100.0


In [4]:
print("Filas totalmente duplicadas:", df.duplicated().sum())
print("IDs de anuncio duplicados:", df["id"].duplicated().sum())

Filas totalmente duplicadas: 0
IDs de anuncio duplicados: 0


In [5]:
# "faltantes disfrazados": valores de texto que en realidad significan vacío pero no son NaN
placeholders = ["N", "NA", "N/A", "n/a", "null", "NULL", "None", "none", "-", "--"]
obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
placeholders_encontrados = {}
for c in obj_cols:
    hits = [v for v in df[c].dropna().unique() if str(v).strip() in placeholders]
    if hits:
        placeholders_encontrados[c] = hits

print("Columnas con placeholders de 'faltante' disfrazados de texto:",
      placeholders_encontrados if placeholders_encontrados else "ninguna")

Columnas con placeholders de 'faltante' disfrazados de texto: {'host_about': ['-']}


**Hallazgos del análisis inicial:**
- El dataset trae 90 columnas y 11.111 anuncios de Austin, sin filas ni IDs duplicados.
- Se revisaron las columnas de texto en busca de "faltantes disfrazados" (`"N"`, `"NA"`, `"-"`, etc. que no son `NaN` pero significan vacío); solo apareció un caso aislado en `host_about`, columna que de todas formas se descarta por ser texto libre sin procesar.
- 14 columnas llegan 100% vacías (p. ej. `host_since`, `host_response_rate`, `bathrooms`, `instant_bookable`, `neighbourhood`), es decir, el campo existe pero nunca se diligenció en esta extracción.
- Varias columnas de identificación y metadatos de scraping (URLs, `scrape_id`, `source`, cotizaciones puntuales de precio) no aportan valor analítico para responder las preguntas de negocio.
- Los puntajes de reseñas (`review_scores_*`) y `reviews_per_month` tienen 16.7% de nulos, que corresponden exactamente a los anuncios sin reseñas todavía (`number_of_reviews == 0`), no a datos faltantes por error.
- `price` (variable objetivo para el modelo de regresión) tiene 10.0% de nulos.

## 2. Selección de variables potencialmente relevantes
Se eliminan: (a) columnas 100% vacías, y (b) columnas técnicas/identificadoras (URLs, IDs internos de scraping, cotizaciones puntuales, `estimated_revenue_l365d` por ser una combinación derivada de precio y ocupación que generaría fuga de información hacia el modelo de precio).

In [6]:
cols_100_null = df.columns[df.isnull().mean() == 1.0].tolist()

cols_tech = [
    "listing_url", "scrape_id", "picture_url", "host_url", "host_profile_id",
    "host_profile_url", "host_picture_url", "calendar_last_scraped",
    "source", "host_id", "host_name",
    "price_quote_checkin_date", "price_quote_checkout_date", "price_quote_raw",
    "price_quote_total_price", "price_quote_price_per_night",
    "estimated_revenue_l365d",
]
cols_tech = [c for c in cols_tech if c in df.columns]
# nota: last_scraped, first_review y last_review no se eliminan aun -> se usan mas abajo
# para calcular la recencia de las reseñas antes de descartar las fechas crudas

print("Columnas 100% nulas eliminadas:", len(cols_100_null))
print("Columnas técnicas/identificadoras eliminadas:", len(cols_tech))

df = df.drop(columns=list(set(cols_100_null + cols_tech)))
df.shape

Columnas 100% nulas eliminadas: 14
Columnas técnicas/identificadoras eliminadas: 17


(11111, 59)